# Compare facility screening with the AIDS 2024 baseline

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/16_facility_comparison.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/16_facility_comparison.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `75cbc6da16fc3e9b389eb6a440a14584b4c2a4fe5e592ec71db63e227b7b731f`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Compare facility screening with the AIDS 2024 baseline

**Time:** 45 minutes · **Prerequisites:** chapters 03–04 or basic Python tables. No private input or GIS installation is required.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/16_facility_comparison.ipynb)

**Reading the book?** It shows saved results. You can zoom and rotate Plotly figures here; Python-backed selectors, sliders and uploads require running the notebook in JupyterLite.

Wait for **Python (Pyodide) | Idle**, then choose **Run → Run All Cells**. Public results below reflect the bundled aggregate artifact; unavailable models are labeled. Fictional exercises remain explicitly separate from the study.

In [ ]:
import sys
from pathlib import Path
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'plotly', 'ipywidgets'])
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from coastlab import show, land_traces
from naturelab import (msl_elevation, screening_summary, paired_comparison,
                       load_public, load_sources, coverage_frame, fictional_samples,
                       aggregate_explorer, sensitivity_controls, terrain_explorer)
public = load_public()
sources = load_sources()
print('Ready: Nature 2026 — public aggregates and explicitly fictional exercises.')
print('Current sampling status:', public['status'])

## Objectives: compare like rows and unlike questions honestly

Reconcile the AIDS 2024 baseline, inspect model-specific coverage, and interpret a matched 2×2 table without calling disagreement an error. No private facility points appear in this chapter.

The historical poster asks about potentially flooded facilities under RCP/year categories. The new screening asks whether sampled terrain is at or below a fixed MSL threshold. No year or RCP is attached to the latter. A value ≤1 m is neither an RCP 8.5 forecast nor a 1% annual chance of flooding.

In [ ]:
poster = pd.read_csv(root / 'data/poster_aggregates.csv')
baseline = poster.loc[poster.scope.eq('portfolio')]
display(baseline.pivot(index='year', columns='scenario', values='potentially_flooded'))
fig = go.Figure()
for scenario, rows in baseline.groupby('scenario'):
    fig.add_scatter(x=rows.year, y=rows.potentially_flooded, mode='lines+markers', name=scenario)
show(fig.update_layout(title='AIDS 2024 published portfolio categories; denominator 25,873',
                       xaxis_title='Projection year', yaxis_title='Published potentially flooded facilities'))
audit = public.get('baseline_audit')
if audit is None:
    print('Current Nature sampling run has not audited the private workbook yet.')
else:
    print('Current workbook: all24 counts match:', audit['all_24_match'],
          '; denominator matches:', audit['denominator_matches'])
    display(pd.DataFrame(audit['counts']))

## Public geographic context

The map below computes 2030 counts from the published **city** rows only. Its points are approximate city anchors; marker size represents a published city aggregate. It provides geographic context and is not a map of sampled facilities or of Nature raster coverage. City and district scopes must never be added together.

In [ ]:
cities = poster.loc[poster.scope.eq('city') & poster.scenario.eq('RCP 8.5') & poster.year.eq(2030)]
fig = go.Figure(land_traces())
fig.add_scatter(x=cities.longitude, y=cities.latitude, mode='markers+text',
                text=cities.city, textposition='top center',
                marker=dict(size=8 + 3*np.sqrt(cities.potentially_flooded), color='#087e8b'),
                customdata=cities[['potentially_flooded', 'facilities_2023']].to_numpy(),
                hovertemplate='%{text}<br>Published: %{customdata[0]} / %{customdata[1]}<extra></extra>')
show(fig.update_layout(title='Published city aggregates, RCP 8.5 / 2030 — contextual city anchors',
                       xaxis_title='Longitude', yaxis_title='Latitude', showlegend=False,
                       xaxis_range=[-25, 125], yaxis_range=[-20, 40], yaxis_scaleanchor='x'))

In [ ]:
display(coverage_frame(public))
public_controls = aggregate_explorer(public)

## Same-row comparison, different references

The primary comparison is the **audited RCP flags** versus static MSL screening. Original `ACTUALE` contains numerous zeros and some very negative numeric values. Zero may mean an unassessed/fill value rather than valid terrain at the reference surface. Its meaning requires source metadata; finite numeric values alone do not establish valid elevations.

The optional original-height audit pairs **the same supported rows with finite original numbers and valid Nature heights**. A 2×2 table then describes raw numeric thresholds only. It is disabled below unless `ENABLE_RAW_ORIGINAL_AUDIT` is explicitly enabled. Zero and extreme-negative counts remain visible in the audit. The original is MHHW-labeled and the new product is MSL: agreement is not accuracy, and disagreement cannot be attributed solely to datum. Terrain measurement, epoch and spatial resolution also change.

RCP cross-tabs likewise describe overlap with RED/ORANGE/YELLOW flags, not sensitivity/specificity against observed flooding. Missing new elevations are excluded from both sides of the paired table and counted as unmatched. The sum of its four cells must equal `matched`.

First inspect **per-model covered cohorts**. A partial regional FABDEM sample and a broader GLL sample may include different rows. Their individual absolute counts cannot establish which model has more exposure. The across-model chart below first intersects the supported rows with valid cells in **every** listed model. It uses this smaller, common denominator for all models and keeps the unmatched portfolio visible. It still does not measure accuracy or establish representativeness beyond that shared subset.

In [ ]:
ENABLE_RAW_ORIGINAL_AUDIT = False
original_audit = public.get('original_elevation_audit')
if original_audit:
    print(original_audit['interpretation'])
    display(pd.DataFrame([{k: v for k, v in original_audit.items() if k != 'interpretation'}]))
common = public.get('common_cohort', [])
if common and common[0]['matched']:
    common_rows = [{'model': model_id, 'threshold_m': item['threshold_m'],
                    'at_or_below': count, 'matched': item['matched'], 'unmatched': item['unmatched']}
                   for item in common for model_id, count in item['counts'].items()]
    common_table = pd.DataFrame(common_rows)
    display(common_table)
    fig = go.Figure()
    for model_id, rows in common_table.groupby('model'):
        metadata = next(m for m in public['models'] if m['id'] == model_id)
        fig.add_bar(x=rows.threshold_m.astype(str), y=rows.at_or_below,
                    name=metadata['dataset'] + ' v' + metadata['version'])
    show(fig.update_layout(barmode='group',
                           title=f"Shared supported cohort: {common[0]['matched']} matched; {common[0]['unmatched']} unmatched",
                           xaxis_title='Static MSL threshold (m)', yaxis_title='Facilities at/below threshold'))
else:
    print('Common-model cohort unavailable; no cross-model count comparison is made.')
for model in public['models']:
    print(model['label'], '—', model['status'])
    paired = next((p for p in model.get('paired_original', []) if p['threshold_m'] == 1), None)
    if not ENABLE_RAW_ORIGINAL_AUDIT:
        print('Raw original numeric audit disabled; original zero/fill semantics are unverified.')
    elif paired is None or not paired['matched']:
        print('Matched original/Nature comparison unavailable; no matrix is invented.')
    else:
        assert sum(paired['cells'].values()) == paired['matched']
        display(pd.DataFrame([{'matched': paired['matched'], 'unmatched': paired['unmatched'], **paired['cells']}]))
        c = paired['cells']
        z = [[c['both_at_or_below'], c['original_only']], [c['nature_only'], c['neither']]]
        show(go.Figure(go.Heatmap(z=z, x=['Nature <=1 m MSL', 'Nature >1 m MSL'],
                                 y=['Original <=1 m MHHW', 'Original >1 m MHHW'],
                                 text=z, texttemplate='%{text}', colorscale='Blues'))
             .update_layout(title=model['label'] + ': RAW UNVERIFIED numeric overlap, not accuracy'))
    overlap = [r for r in model.get('rcp_overlap', []) if r['year'] == 2030 and r['threshold_m'] == 1]
    if overlap:
        display(pd.DataFrame([{'scenario': r['scenario'], 'matched': r['matched'], **r['cells']} for r in overlap]))

## Lab: interpret the off-diagonal cells

In an invented matched cohort of 100, both methods flag 12 rows, original alone flags 8, Nature alone flags 20, and neither flags 60. How many are matched? Which method is more accurate? What changes if 25 additional cohort rows lack Nature elevations?

**Worked answer:** The matrix conserves 100 rows. Accuracy is unknown because the labels answer different questions and neither is observed truth. Original flags 20/100 and Nature flags 32/100 in the matched subset. With 25 unmatched rows, coverage is 100/125=80%; the four observed cells do not change. Never silently classify those 25 as “neither.”

**Deliverable:** A comparison paragraph containing the model/version, fixed threshold, covered and full denominators, paired cohort, datum warning and one additional input needed for a flood assessment (for example local water levels, hydraulic connectivity or defenses).

## Sources and evidence boundaries

Seeger & Minderhoud (2026), [*Sea level much higher than assumed in most coastal hazard assessments*](https://doi.org/10.1038/s41586-026-10196-1), with the [25 August 2026 addendum](https://doi.org/10.1038/s41586-026-11017-1). The pinned [October 2026 catalog, record 23341921](https://doi.org/10.5281/zenodo.23341921) describes corrected products and their licenses. Exact versions, URLs, roles and inspected metadata hashes are in `data/nature-sources.json`; current results are in `data/nature-aggregates.json`.

Original calculations/code here are independent MIT implementations. Corrected-raster derivatives are **CC BY-NC-SA 4.0**, distinct from the **CC BY 4.0** original lesson prose and paper. Cite the corrected product, underlying DEM and MDT/geoid sources. No private workbook, facility identifiers, names, coordinates or clinical counts are bundled. [Full source audit](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/source-audit.html) · [Data dictionary](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/data-dictionary.html) · [References](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/references.html).

In [ ]:
print('Nature lesson complete: 16_facility_comparison')